# Practical Exercise with MNIST Example

In [1]:
import torch # 파이토치 임포트
import torch.nn # 신경망 모듈 임포트

In [2]:
import sys # 시스템 모듈
import numpy as np # 넘파이 임포트
import matplotlib.pyplot as plt # 시각화 도구

from model import ImageClassifier # 사용자 정의 모델

from utils import load_mnist # 데이터 로드 함수
from utils import split_data # 데이터 분리 함수
from utils import get_hidden_sizes # 은닉층 크기 계산

In [ ]:
model_fn = "./model.pth" # 불러올 모델 파일 경로

In [ ]:
# GPU 사용 가능 여부에 따라 장치 설정
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
def load(fn, device):
    # 저장된 파일에서 모델 상태와 설정을 로드
    d = torch.load(fn, map_location=device)
    
    return d['model'], d['config']

In [ ]:
def plot(x, y_hat):
    # 예측 결과를 이미지와 함께 출력하는 함수
    for i in range(x.size(0)):
        # 텐서를 넘파이 배열로 변환하고 28x28로 재구성
        img = (np.array(x[i].detach().cpu(), dtype='float')).reshape(28,28)

        plt.imshow(img, cmap='gray') # 흑백 이미지 출력
        plt.show()
        # 가장 높은 확률의 인덱스를 예측값으로 출력
        print("Predict:", float(torch.argmax(y_hat[i], dim=-1)))

In [ ]:
def test(model, x, y, to_be_shown=True):
    # 모델 평가 수행 함수
    model.eval() # 평가 모드 설정
    
    with torch.no_grad(): # 기울기 계산 비활성화
        y_hat = model(x) # 예측

        # 맞은 개수 계산
        correct_cnt = (y.squeeze() == torch.argmax(y_hat, dim=-1)).sum()
        total_cnt = float(x.size(0))
        
        accuracy = correct_cnt / total_cnt # 정확도 계산
        print("Accuracy: %.4f" % accuracy)
        
        if to_be_shown:
            plot(x, y_hat) # 시각화 수행

In [ ]:
# 모델 상태와 설정 로드
model_dict, train_config = load(model_fn, device)

# MNIST 테스트 데이터 로드
x, y = load_mnist(is_train=False)
x, y = x.to(device), y.to(device)

input_size = int(x.shape[-1])
output_size = int(max(y)) + 1

# 로드된 설정을 바탕으로 모델 구조 생성
model = ImageClassifier(
    input_size=input_size,
    output_size=output_size,
    hidden_sizes=get_hidden_sizes(input_size,
                                  output_size,
                                  train_config.n_layers),
    use_batch_norm=not train_config.use_dropout,
    dropout_p=train_config.dropout_p,
).to(device)

model.load_state_dict(model_dict) # 가중치 로드

# 테스트 수행 (정확도만 확인)
test(model, x, y, to_be_shown=False)

In [ ]:
# 일부 데이터에 대해 시각화와 함께 예측 수행
n_test = 20
test(model, x[:n_test], y[:n_test], to_be_shown=True)